# Un autoencoder por dentro

**Ampliación autoral de Hespérides · Capítulo 4**

Reutilizamos el patrón MLP de `locked/chapter_multilayer-perceptrons/mlp-implementation.ipynb`: capas afines y activaciones compuestas con `nn.Sequential`. La tarea de reconstrucción y los datos sintéticos son nuevos. No es una traducción de un supuesto notebook de autoencoders en `locked`.

Un encoder transforma 256 píxeles en dos números; un decoder intenta reconstruir la imagen a partir de ellos. La pérdida compara la reconstrucción con la entrada. Aquí la familia de imágenes depende principalmente de dos coordenadas —la posición de una mancha—, de modo que el cuello de botella tiene una justificación concreta. Esto no demuestra que dos números puedan preservar cualquier imagen.

Primero inspecciona una imagen y su reconstrucción. Después recorre el espacio latente. Lejos de los puntos vistos, el decoder extrapola sin garantía de producir una muestra válida. Las dos coordenadas aprendidas no tienen por qué coincidir con las coordenadas físicas ni tener una interpretación única.


In [ ]:
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
import ipywidgets as widgets
from ipywidgets import interact

torch.manual_seed(42)
np.random.seed(42)
torch.set_num_threads(2)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False,
                     "axes.spines.right": False, "animation.embed_limit": 40})


In [ ]:

# Familia sintética bidimensional de imágenes: una mancha cambia de posición.
g = torch.Generator().manual_seed(7)
centros = 1.4*torch.rand((320,2), generator=g)-.7
coord = torch.linspace(-1,1,16)
gy,gx = torch.meshgrid(coord,coord,indexing='ij')
imagenes = torch.exp(-((gx[None]-centros[:,0,None,None])**2 +
                       (gy[None]-centros[:,1,None,None])**2)/.075)
entradas=imagenes.flatten(1)
encoder=nn.Sequential(nn.Linear(256,64),nn.ReLU(),nn.Linear(64,2))
decoder=nn.Sequential(nn.Linear(2,64),nn.ReLU(),nn.Linear(64,256),nn.Sigmoid())
opt=torch.optim.Adam(list(encoder.parameters())+list(decoder.parameters()),lr=.008)
curva=[]
for epoca in range(450):
    opt.zero_grad()
    reconstruccion=decoder(encoder(entradas[:256]))
    perdida=nn.functional.mse_loss(reconstruccion,entradas[:256])
    perdida.backward();opt.step();curva.append(perdida.item())
with torch.no_grad():
    Z=encoder(entradas)
    reconstruidas=decoder(Z).reshape(-1,16,16)
    media=Z[:256].mean(0);escala=Z[:256].std(0).clamp_min(1e-6)
    Zn=(Z-media)/escala
    mse_val=nn.functional.mse_loss(reconstruidas[256:],imagenes[256:]).item()
print(f'MSE final de validación: {mse_val:.5f} · 64 imágenes no usadas para ajustar')

def ver_reconstruccion(indice=280):
    fig,ax=plt.subplots(1,4,figsize=(13,3))
    ax[0].imshow(imagenes[indice],cmap='magma',vmin=0,vmax=1);ax[0].set_title('Original')
    ax[1].imshow(reconstruidas[indice],cmap='magma',vmin=0,vmax=1);ax[1].set_title('Reconstrucción')
    ax[2].imshow((reconstruidas[indice]-imagenes[indice]).abs(),cmap='magma',vmin=0,vmax=.3);ax[2].set_title('Error absoluto')
    ax[3].scatter(*Zn.T,c=centros[:,0],cmap='viridis',s=9)
    ax[3].scatter(*Zn[indice],facecolor='none',edgecolor='red',s=120)
    ax[3].set(title='Dos coordenadas latentes',xlabel='z₁ estandarizada',ylabel='z₂ estandarizada')
    for a in ax[:3]:a.axis('off')
    fig.tight_layout();plt.show()
interact(ver_reconstruccion,indice=widgets.IntSlider(value=280,min=0,max=319,
                                                   description='Imagen',continuous_update=False));

def recorrer_latente(z1=0.,z2=0.):
    with torch.no_grad():
        z=torch.tensor([z1,z2])*escala+media
        imagen_decodificada=decoder(z).reshape(16,16)
    fig,ax=plt.subplots(1,2,figsize=(7,3))
    ax[0].scatter(*Zn[:256].T,c=centros[:256,0],cmap='viridis',s=10)
    ax[0].scatter(z1,z2,color='red',marker='x',s=100)
    ax[0].set(xlim=(-3,3),ylim=(-3,3),title='Consulta en el espacio latente',xlabel='z₁',ylabel='z₂')
    ax[1].imshow(imagen_decodificada,cmap='magma',vmin=0,vmax=1);ax[1].set_title('Salida del decoder');ax[1].axis('off')
    fig.tight_layout();plt.show()
interact(recorrer_latente,
         z1=widgets.FloatSlider(value=0,min=-3,max=3,step=.1,continuous_update=False),
         z2=widgets.FloatSlider(value=0,min=-3,max=3,step=.1,continuous_update=False));


In [ ]:
ver_reconstruccion(280)
recorrer_latente(0.,0.)

## Del autoencoder al VAE

Un autoencoder determinista optimiza reconstrucción. Un VAE introduce una distribución latente y un objetivo variacional que equilibra reconstrucción y divergencia respecto a una prior. El ejemplo anterior **no es un VAE**: no tiene muestreo mediante reparametrización ni término KL. La derivación de VAE queda vinculada a los apuntes de la sesión 4, no sustituida por esta demostración.

## Preguntas

1. ¿Por qué un espacio latente de dimensión dos puede funcionar para esta familia de imágenes y no para cualquier conjunto de fotografías?
2. ¿Por qué una reconstrucción convincente no demuestra que las coordenadas latentes sean interpretables o únicas?
3. ¿Qué riesgo tiene mover los controles lejos de los puntos latentes observados?
4. ¿Qué elementos faltan para que este modelo sea un VAE?
